In [4]:
%%writefile fullalignment.c

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>
#include <time.h>

// 24 bytes per timer
typedef struct {
    uint64_t start_cycles;  // 8 bytes
    uint64_t total_cycles;  // 8 bytes
    uint64_t call_count;    // 8 bytes (using uint64_t to match 64-bit ASM long)
} Timer;

// Layout matching ASM offsets:
// 0:  preprocess
// 24: main_diagonal
// 48: right_diagonal
// 72: left_diagonal
// 96: snake_total
typedef struct {
    Timer preprocess_timer;       // Offset 0
    Timer main_diagonal_timer;    // Offset 24
    Timer right_diagonal_timer;   // Offset 48
    Timer left_diagonal_timer;    // Offset 72
    Timer snake_total_timer;      // Offset 96
} TimingData;

// Global instance shared with ASM
TimingData timing_data = {0};

// Helper to convert CPU cycles to seconds (Assuming 4GHz CPU, adjust if needed)
static double cycles_to_seconds(uint64_t cycles) {
    static double cpu_freq_ghz = 4.0; 
    return cycles / (cpu_freq_ghz * 1e9);
}

void print_timing_breakdown() {
    printf("\n=== ASM INTERNAL TIMING BREAKDOWN ===\n");
    printf("%-25s %12s %12s %15s\n", "Component", "Total (s)", "Calls", "Avg Cycles");
    printf("----------------------------------------------------------------------------\n");

    #define PRINT_TIMER(name, t) \
        printf("%-25s %12.6f %12lu %15.0f\n", \
               name, \
               cycles_to_seconds(t.total_cycles), \
               t.call_count, \
               t.call_count > 0 ? (double)t.total_cycles / t.call_count : 0.0)

    PRINT_TIMER("Total SneakySnake", timing_data.snake_total_timer);
    PRINT_TIMER("  Main Diagonal",   timing_data.main_diagonal_timer);
    PRINT_TIMER("  Right (Upper) Diag", timing_data.right_diagonal_timer);
    PRINT_TIMER("  Left (Lower) Diag",  timing_data.left_diagonal_timer);
    
    // Calculate overhead (Total - components)
    uint64_t components = timing_data.main_diagonal_timer.total_cycles +
                          timing_data.right_diagonal_timer.total_cycles +
                          timing_data.left_diagonal_timer.total_cycles;
    
    if (timing_data.snake_total_timer.total_cycles > components) {
        uint64_t overhead = timing_data.snake_total_timer.total_cycles - components;
        printf("%-25s %12.6f %12s %15s\n", 
               "  Other/Overhead", cycles_to_seconds(overhead), "-", "-");
    }
    printf("----------------------------------------------------------------------------\n");
}

void reset_asm_timers() {
    memset(&timing_data, 0, sizeof(TimingData));
}

// ==============================================================================
// ALIGNMENT LOGIC
// ==============================================================================

void preprocess(char* RefSeq, char* ReadSeq, int ReadLength)
{
    for (int i = 0; i < ReadLength; i++) {
        char r = ReadSeq[i];
        char f = RefSeq[i];
        
        ReadSeq[i] = (r == 'A' || r == 'a') ? 1 :
                     (r == 'C' || r == 'c') ? 2 :
                     (r == 'G' || r == 'g') ? 3 :
                     (r == 'T' || r == 't') ? 4 :
                     (r == 'N' || r == 'n') ? 5 : 0;
                      
        RefSeq[i] = (f == 'A' || f == 'a') ? 1 :
                    (f == 'C' || f == 'c') ? 2 :
                    (f == 'G' || f == 'g') ? 3 :
                    (f == 'T' || f == 't') ? 4 :
                    (f == 'N' || f == 'n') ? 5 : 0;
    }
    
    int remainder = ReadLength % 64;
    if (remainder != 0) {
        int padding = 64 - remainder;
        for (int i = 0; i < padding; i++) {
            ReadSeq[ReadLength + i] = 0xFF;
            RefSeq[ReadLength + i] = 0x00;
        }
    }
}

// Original simple C implementation
int SneakySnake_C(int ReadLength, uint8_t* RefSeq, uint8_t* ReadSeq, 
                  int EditThreshold, int IterationNo)
{
    int index = 0;
    int Edits = 0;
    int roundsNo = 1;
    
    while (index < ReadLength) {
        if (roundsNo > IterationNo) return 0;
        if (Edits > EditThreshold) return 0;
        
        int GlobalCount = 0;
        int count = 0;
        
        // Main diagonal
        for (int n = index; n < ReadLength; n++) {
            if (ReadSeq[n] != RefSeq[n]) break;
            count++;
        }
        GlobalCount = count;
        
        if (GlobalCount == (ReadLength - index)) {
            return (Edits <= EditThreshold) ? 1 : 0;
        }
        
        // Upper and lower diagonals
        for (int e = 1; e <= EditThreshold; e++) {
            count = 0;
            // Upper diagonal
            for (int n = index; n < ReadLength; n++) {
                if (n < e) break;
                if (ReadSeq[n - e] != RefSeq[n]) break;
                count++;
            }
            if (count > GlobalCount) GlobalCount = count;
            if (count == (ReadLength - index)) return (Edits <= EditThreshold) ? 1 : 0;
            
            count = 0;
            // Lower diagonal
            for (int n = index; n < ReadLength; n++) {
                if (n > ReadLength - e - 1) break;
                if (ReadSeq[n + e] != RefSeq[n]) break;
                count++;
            }
            if (count > GlobalCount) GlobalCount = count;
            if (count == (ReadLength - index)) return (Edits <= EditThreshold) ? 1 : 0;
        }
        
        index += GlobalCount;
        if (index < ReadLength) {
            Edits++;
            index++;
        }
        roundsNo++;
    }
    
    return (Edits <= EditThreshold) ? 1 : 0;
}

// External ASM function
extern uint64_t SneakySnake(uint64_t ReadLength, uint8_t* RefSeq, 
                            uint8_t* ReadSeq, uint64_t EditThreshold, 
                            uint64_t IterationNo);
extern uint64_t current_position, current_edits, mismatch_count, safety_counter;

int calculate_edit_distance(const char* s1, const char* s2, int len) {
    if (len > 2000) return 0; 
    int m = len, n = len;
    int dp[m + 1][n + 1];
    for (int i = 0; i <= m; i++) dp[i][0] = i;
    for (int j = 0; j <= n; j++) dp[0][j] = j;
    for (int i = 1; i <= m; i++) {
        for (int j = 1; j <= n; j++) {
            if (s1[i-1] == s2[j-1]) {
                dp[i][j] = dp[i-1][j-1];
            } else {
                dp[i][j] = 1 + (dp[i-1][j] < dp[i][j-1] ? 
                               (dp[i-1][j] < dp[i-1][j-1] ? dp[i-1][j] : dp[i-1][j-1]) :
                               (dp[i][j-1] < dp[i-1][j-1] ? dp[i][j-1] : dp[i-1][j-1]));
            }
        }
    }
    return dp[m][n];
}

int main(int argc, char *argv[]) {
    if (argc < 4) {
        printf("Usage: %s <file> <threshold> <seq_len> [limit] [debug_limit]\n", argv[0]);
        return 1;
    }
    
    const char* filename = argv[1];
    int threshold = atoi(argv[2]);
    int fixed_len = atoi(argv[3]); 
    int limit = (argc >= 5) ? atoi(argv[4]) : 30000;
    int debug_limit = (argc >= 6) ? atoi(argv[5]) : 5;
    
    FILE *file = fopen(filename, "r");
    if (!file) {
        printf("Error: Cannot open file %s\n", filename);
        return 1;
    }
    
    printf("Loading sequences (Fixed Len=%d, Limit=%d)...\n", fixed_len, limit);
    
    char **read_orig = malloc(limit * sizeof(char*));
    char **ref_orig = malloc(limit * sizeof(char*));
    uint8_t **read_enc = malloc(limit * sizeof(uint8_t*));
    uint8_t **ref_enc = malloc(limit * sizeof(uint8_t*));
    int *lengths = malloc(limit * sizeof(int));
    int *edit_dists = malloc(limit * sizeof(int));
    
    char *line = NULL;
    size_t cap = 0;
    int count = 0;
    int ground_truth_matches = 0;
    int skipped = 0;
    
    while (getline(&line, &cap, file) != -1 && count < limit) {
        char *read_seq = line;
        char *ref_seq = strpbrk(line, "\t ");
        if (!ref_seq) continue;
        *ref_seq = '\0';
        ref_seq++;
        while (*ref_seq == ' ' || *ref_seq == '\t') ref_seq++;
        ref_seq[strcspn(ref_seq, "\r\n")] = 0;
        
        int actual_len = strlen(read_seq);
        if (actual_len < fixed_len) {
            skipped++;
            continue; 
        }
        
        int len = fixed_len;
        read_seq[len] = '\0';
        ref_seq[len] = '\0';
        
        lengths[count] = len;
        read_orig[count] = strdup(read_seq);
        ref_orig[count] = strdup(ref_seq);
        
        read_enc[count] = calloc(len + 64, 1);
        ref_enc[count] = calloc(len + 64, 1);
        
        memcpy(read_enc[count], read_seq, len);
        memcpy(ref_enc[count], ref_seq, len);
        preprocess((char*)ref_enc[count], (char*)read_enc[count], len);
        
        edit_dists[count] = calculate_edit_distance(read_seq, ref_seq, len);
        if (edit_dists[count] <= threshold && len <= 2000) {
            ground_truth_matches++;
        }
        
        count++;
        if (count % 500 == 0) {
            printf("Loaded %d sequences...\r", count);
            fflush(stdout);
        }
    }
    fclose(file);
    free(line);
    
    // Debug first few (Timers disabled here to avoid polluting benchmark stats)
    printf("\n=== DEBUGGING FIRST %d SEQUENCES ===\n", debug_limit);
    
    for (int i = 0; i < debug_limit && i < count; i++) {
        int c_result = SneakySnake_C(lengths[i], ref_enc[i], read_enc[i], 
                                     threshold, lengths[i] * 2);
        
        // Reset counters/timers just for debug clarity
        current_position = 0;
        current_edits = 0;
        mismatch_count = 0;
        safety_counter = 0;
        
        int asm_result = SneakySnake(lengths[i], ref_enc[i], read_enc[i], 
                                     threshold, lengths[i] * 2);
        
        printf("\n[%d] Len=%d\n", i, lengths[i]);
        printf("    C:    %s\n", c_result ? "ACCEPT" : "REJECT");
        printf("    ASM:  %s\n", asm_result ? "ACCEPT" : "REJECT");
        
        if (c_result != asm_result) {
            printf("    *** C AND ASM DISAGREE! ***\n");
        }
    }
    
    // Clear timers before full benchmark
    reset_asm_timers();
    
    // Full benchmark
    printf("\n=== RUNNING FULL BENCHMARK ===\n");
    int c_matches = 0;
    int asm_matches = 0;
    
    // C implementation
    clock_t c_start = clock();
    for (int i = 0; i < count; i++) {
        int result = SneakySnake_C(lengths[i], ref_enc[i], read_enc[i], 
                                   threshold, lengths[i] * 2);
        if (result) c_matches++;
    }
    clock_t c_end = clock();
    double c_time = ((double)(c_end - c_start)) / CLOCKS_PER_SEC;
    
    // ASM implementation (With Timing Instrumentation)
    clock_t asm_start = clock();
    for (int i = 0; i < count; i++) {
        current_position = 0;
        current_edits = 0;
        mismatch_count = 0;
        safety_counter = 0;
        
        int result = SneakySnake(lengths[i], ref_enc[i], read_enc[i], 
                                 threshold, lengths[i] * 2);
        if (result) asm_matches++;
    }
    clock_t asm_end = clock();
    double asm_time = ((double)(asm_end - asm_start)) / CLOCKS_PER_SEC;
    
    printf("\n==================================================\n");
    printf("                  BENCHMARK SUMMARY               \n");
    printf("==================================================\n");
    printf("Total Pairs              : %d\n", count);
    printf("Fixed Length             : %d\n", fixed_len);
    printf("Threshold                : %d\n", threshold);
    printf("\n");
    printf("C Implementation:\n");
    printf("  Accepted               : %d\n", c_matches);
    printf("  Time                   : %.4f seconds\n", c_time);
    printf("\n");
    printf("ASM Implementation:\n");
    printf("  Accepted               : %d\n", asm_matches);
    printf("  Time                   : %.4f seconds\n", asm_time);
    if (asm_time > 0)
        printf("  Speedup vs C           : %.2fx\n", c_time / asm_time);
    
    // NEW: Print detailed breakdown
    print_timing_breakdown();
    
    if (c_matches != asm_matches) {
        printf("\nWARNING: C and ASM implementations disagree!\n");
        printf("C accepted %d, ASM accepted %d (diff = %d)\n", 
               c_matches, asm_matches, abs(c_matches - asm_matches));
    }
    
    // Cleanup
    for (int i = 0; i < count; i++) {
        free(read_orig[i]);
        free(ref_orig[i]);
        free(read_enc[i]);
        free(ref_enc[i]);
    }
    free(read_orig);
    free(ref_orig);
    free(read_enc);
    free(ref_enc);
    free(lengths);
    free(edit_dists);
    
    return 0;
}

Overwriting fullalignment.c


In [5]:
%%writefile fullalignment.asm

default rel
bits 64

; ==============================================================================
; TIMING MACROS
; ==============================================================================
; Structure offsets based on C definition (Timer struct = 24 bytes)
%define OFF_MAIN_DIAG  24
%define OFF_RIGHT_DIAG 48
%define OFF_LEFT_DIAG  72
%define OFF_TOTAL      96

; Macro to start a timer
%macro START_TIMER 1
    push    rax
    push    rdx
    rdtsc                   ; Time in EDX:EAX
    shl     rdx, 32
    or      rax, rdx        ; Combine to 64-bit in RAX
    mov     %1, rax         ; Save start time to stack
    pop     rdx
    pop     rax
%endmacro

; Macro to stop a timer and accumulate results
%macro STOP_TIMER 2
    push    rax
    push    rdx
    push    rcx             ; Save RCX (often used as temp)
    
    rdtsc                   ; Get end time
    shl     rdx, 32
    or      rax, rdx
    
    sub     rax, %1         ; RAX = End - Start (Delta)
    
    ; Load address of timing_data
    lea     rcx, [timing_data + %2]
    
    ; Add delta to total_cycles (offset 8 in Timer struct)
    add     [rcx + 8], rax
    
    ; Increment call_count (offset 16 in Timer struct)
    inc     qword [rcx + 16]
    
    pop     rcx
    pop     rdx
    pop     rax
%endmacro

section .data
global SneakySnake
global current_position
global current_edits
global mismatch_count
global safety_counter

; Import the C struct
extern timing_data

align 64
current_position dq 0
current_edits dq 0
mismatch_count dq 0
safety_counter dq 0

section .text
global SneakySnake

SneakySnake:
    push    rbp
    mov     rbp, rsp
    push    rbx
    push    r12
    push    r13
    push    r14
    push    r15
    sub     rsp, 64 
    ; Stack Map for Local Variables:
    ; [rbp-8]  : IterationNo
    ; [rbp-16] : Edits
    ; [rbp-24] : roundsNo
    ; [rbp-32] : Main Diag Start Time
    ; [rbp-40] : Right Diag Start Time
    ; [rbp-48] : Left Diag Start Time
    ; [rbp-56] : Total Function Start Time
    
    ; --- START TOTAL TIMER ---
    START_TIMER [rbp-56]

    mov     r13, rdi              ; ReadLength
    mov     r12, rsi              ; RefSeq
    mov     r11, rdx              ; ReadSeq
    mov     r10, rcx              ; EditThreshold
    mov     [rbp-8], r8           ; IterationNo

    ; Initialize counters
    xor     rax, rax
    mov     [current_position], rax
    mov     [current_edits], rax
    mov     [mismatch_count], rax
    mov     [safety_counter], rax

    xor     r15, r15              ; index = 0
    mov     qword [rbp-16], 0     ; Edits = 0
    mov     qword [rbp-24], 1     ; roundsNo = 1
    
.while_loop:
    cmp     r15, r13
    jae     .accept
    
    mov     rax, [rbp-24]
    cmp     rax, [rbp-8]
    ja      .reject
    
    mov     r14, [rbp-16]
    cmp     r14, r10
    jg      .reject
    
    ; --- START MAIN DIAGONAL TIMER ---
    START_TIMER [rbp-32]

    ; Check main diagonal with AVX-512 and prefetching
    xor     rbx, rbx              ; GlobalCount = 0
    
    mov     rax, r15
    mov     rcx, r13
    sub     rcx, rax              ; remaining = ReadLength - index
    
    ; Prefetch data for better cache performance
    prefetcht0 [r11 + rax + 64]
    prefetcht0 [r12 + rax + 64]
    
    ; Process 256 bytes at a time (4x ZMM registers) for better throughput
.main_loop_avx512_unrolled:
    cmp     rcx, 256
    jb      .main_loop_avx512_single
    
    ; Prefetch next cache lines
    prefetcht0 [r11 + rax + 320]
    prefetcht0 [r12 + rax + 320]
    
    ; Load and compare 4x64 bytes (unrolled)
    vmovdqu8 zmm0, [r11 + rax]
    vmovdqu8 zmm1, [r12 + rax]
    vmovdqu8 zmm2, [r11 + rax + 64]
    vmovdqu8 zmm3, [r12 + rax + 64]
    
    vpcmpeqb k1, zmm0, zmm1
    vpcmpeqb k2, zmm2, zmm3
    
    ; Check first 64 bytes
    kmovq   rdx, k1
    not     rdx
    test    rdx, rdx
    jnz     .found_mismatch_0
    
    ; Check second 64 bytes
    kmovq   rdx, k2
    not     rdx
    test    rdx, rdx
    jnz     .found_mismatch_64
    
    ; Load and compare next two blocks
    vmovdqu8 zmm4, [r11 + rax + 128]
    vmovdqu8 zmm5, [r12 + rax + 128]
    vmovdqu8 zmm6, [r11 + rax + 192]
    vmovdqu8 zmm7, [r12 + rax + 192]
    
    vpcmpeqb k3, zmm4, zmm5
    vpcmpeqb k4, zmm6, zmm7
    
    ; Check third 64 bytes
    kmovq   rdx, k3
    not     rdx
    test    rdx, rdx
    jnz     .found_mismatch_128
    
    ; Check fourth 64 bytes
    kmovq   rdx, k4
    not     rdx
    test    rdx, rdx
    jnz     .found_mismatch_192
    
    ; All 256 bytes matched
    add     rbx, 256
    add     rax, 256
    sub     rcx, 256
    jmp     .main_loop_avx512_unrolled

.found_mismatch_0:
    tzcnt   rdx, rdx
    add     rbx, rdx
    jmp     .main_done

.found_mismatch_64:
    tzcnt   rdx, rdx
    add     rbx, 64
    add     rbx, rdx
    jmp     .main_done

.found_mismatch_128:
    tzcnt   rdx, rdx
    add     rbx, 128
    add     rbx, rdx
    jmp     .main_done

.found_mismatch_192:
    tzcnt   rdx, rdx
    add     rbx, 192
    add     rbx, rdx
    jmp     .main_done
    
.main_loop_avx512_single:
    cmp     rcx, 64
    jb      .main_loop_scalar
    
    vmovdqu8 zmm0, [r11 + rax]
    vmovdqu8 zmm1, [r12 + rax]
    
    vpcmpeqb k1, zmm0, zmm1
    kmovq   rdx, k1
    not     rdx
    
    test    rdx, rdx
    jz      .all_64_matched
    
    tzcnt   rdx, rdx
    add     rbx, rdx
    jmp     .main_done
    
.all_64_matched:
    add     rbx, 64
    add     rax, 64
    sub     rcx, 64
    jmp     .main_loop_avx512_single
    
.main_loop_scalar:
    ; --------------------------------------------------------
    ; AVX-512 MASKED TAIL (Replaces Scalar Loop)
    ; --------------------------------------------------------
    test    rcx, rcx
    jz      .main_done
    
    mov     rdx, -1
    bzhi    rdx, rdx, rcx       ; Create mask for remaining bytes
    kmovq   k1, rdx
    
    vmovdqu8 zmm0 {k1}{z}, [r11 + rax]
    vmovdqu8 zmm1 {k1}{z}, [r12 + rax]
    
    vpcmpeqb k2 {k1}, zmm0, zmm1
    
    kmovq   r8, k2
    not     r8
    and     r8, rdx             ; Clear garbage bits
    
    test    r8, r8
    jz      .main_tail_match
    
    tzcnt   r8, r8
    add     rbx, r8
    jmp     .main_done
    
.main_tail_match:
    add     rbx, rcx
    jmp     .main_done
    
.main_done:
    ; --- STOP MAIN DIAGONAL TIMER ---
    STOP_TIMER [rbp-32], OFF_MAIN_DIAG

    ; Check if we matched everything
    mov     rcx, r13
    sub     rcx, r15
    cmp     rbx, rcx
    jae     .accept
    
    ; Check shifted diagonals
    mov     r8, 1                 ; shift = 1
    
.diag_loop:
    cmp     r8, r10
    ja      .diag_done
    
    ; --- START RIGHT (UPPER) DIAGONAL TIMER ---
    START_TIMER [rbp-40]

    ; Upper diagonal - AVX-512 optimized with unrolling
    xor     rax, rax              ; count = 0
    mov     r9, r15               ; ref_pos
    
.upper_loop_avx512:
    cmp     r9, r8
    jb      .upper_done
    
    mov     rsi, r9
    sub     rsi, r8
    cmp     rsi, r13
    jae     .upper_done
    
    ; Calculate safe length
    mov     rdx, r13
    sub     rdx, r9
    mov     rdi, r13
    sub     rdi, rsi
    cmp     rdx, rdi
    cmovg   rdx, rdi
    
    ; Process 128 bytes at a time for upper diagonal
    cmp     rdx, 128
    jb      .upper_single_vector
    
    prefetcht0 [r11 + rsi + 128]
    prefetcht0 [r12 + r9 + 128]
    
    vmovdqu8 zmm0, [r11 + rsi]
    vmovdqu8 zmm1, [r12 + r9]
    vmovdqu8 zmm2, [r11 + rsi + 64]
    vmovdqu8 zmm3, [r12 + r9 + 64]
    
    vpcmpeqb k1, zmm0, zmm1
    vpcmpeqb k2, zmm2, zmm3
    
    kmovq   rdx, k1
    not     rdx
    test    rdx, rdx
    jnz     .upper_mismatch_0
    
    kmovq   rdx, k2
    not     rdx
    test    rdx, rdx
    jnz     .upper_mismatch_64
    
    add     rax, 128
    add     r9, 128
    jmp     .upper_loop_avx512

.upper_mismatch_0:
    tzcnt   rdx, rdx
    add     rax, rdx
    jmp     .upper_done

.upper_mismatch_64:
    tzcnt   rdx, rdx
    add     rax, 64
    add     rax, rdx
    jmp     .upper_done
    
.upper_single_vector:
    cmp     rdx, 64
    jb      .upper_scalar
    
    vmovdqu8 zmm0, [r11 + rsi]
    vmovdqu8 zmm1, [r12 + r9]
    
    vpcmpeqb k1, zmm0, zmm1
    kmovq   rdx, k1
    not     rdx
    
    test    rdx, rdx
    jz      .upper_all_64_matched
    
    tzcnt   rdx, rdx
    add     rax, rdx
    jmp     .upper_done
    
.upper_all_64_matched:
    add     rax, 64
    add     r9, 64
    jmp     .upper_loop_avx512
    
.upper_scalar:
    ; --------------------------------------------------------
    ; AVX-512 MASKED TAIL (Replaces Scalar Loop)
    ; --------------------------------------------------------
    ; rdx already holds the safe length from the calculation above
    test    rdx, rdx
    jz      .upper_done
    
    mov     r14, -1             ; Use r14 as temporary (rdx holds length)
    bzhi    r14, r14, rdx
    kmovq   k1, r14
    
    vmovdqu8 zmm0 {k1}{z}, [r11 + rsi]
    vmovdqu8 zmm1 {k1}{z}, [r12 + r9]
    
    vpcmpeqb k2 {k1}, zmm0, zmm1
    
    kmovq   r14, k2
    not     r14
    bzhi    r14, r14, rdx       ; Clean garbage bits
    
    test    r14, r14
    jz      .upper_tail_match
    
    tzcnt   r14, r14
    add     rax, r14
    jmp     .upper_done
    
.upper_tail_match:
    add     rax, rdx
    jmp     .upper_done
    
.upper_done:
    ; --- STOP RIGHT (UPPER) DIAGONAL TIMER ---
    STOP_TIMER [rbp-40], OFF_RIGHT_DIAG

    cmp     rax, rbx
    jbe     .check_lower
    mov     rbx, rax
    
    mov     rcx, r13
    sub     rcx, r15
    cmp     rbx, rcx
    jae     .accept
    
.check_lower:
    ; --- START LEFT (LOWER) DIAGONAL TIMER ---
    START_TIMER [rbp-48]

    ; Lower diagonal - AVX-512 optimized with unrolling
    xor     rax, rax
    mov     r9, r15
    
.lower_loop_avx512:
    lea     rsi, [r9 + r8]
    cmp     rsi, r13
    jae     .lower_done
    
    mov     rdx, r13
    sub     rdx, r8
    cmp     r9, rdx
    jae     .lower_done
    
    ; Calculate safe length
    mov     rdx, r13
    sub     rdx, r9
    mov     rdi, r13
    sub     rdi, rsi
    cmp     rdx, rdi
    cmovg   rdx, rdi
    
    ; Process 128 bytes at a time
    cmp     rdx, 128
    jb      .lower_single_vector
    
    prefetcht0 [r11 + rsi + 128]
    prefetcht0 [r12 + r9 + 128]
    
    vmovdqu8 zmm0, [r11 + rsi]
    vmovdqu8 zmm1, [r12 + r9]
    vmovdqu8 zmm2, [r11 + rsi + 64]
    vmovdqu8 zmm3, [r12 + r9 + 64]
    
    vpcmpeqb k1, zmm0, zmm1
    vpcmpeqb k2, zmm2, zmm3
    
    kmovq   rdx, k1
    not     rdx
    test    rdx, rdx
    jnz     .lower_mismatch_0
    
    kmovq   rdx, k2
    not     rdx
    test    rdx, rdx
    jnz     .lower_mismatch_64
    
    add     rax, 128
    add     r9, 128
    jmp     .lower_loop_avx512

.lower_mismatch_0:
    tzcnt   rdx, rdx
    add     rax, rdx
    jmp     .lower_done

.lower_mismatch_64:
    tzcnt   rdx, rdx
    add     rax, 64
    add     rax, rdx
    jmp     .lower_done
    
.lower_single_vector:
    cmp     rdx, 64
    jb      .lower_scalar
    
    vmovdqu8 zmm0, [r11 + rsi]
    vmovdqu8 zmm1, [r12 + r9]
    
    vpcmpeqb k1, zmm0, zmm1
    kmovq   rdx, k1
    not     rdx
    
    test    rdx, rdx
    jz      .lower_all_64_matched
    
    tzcnt   rdx, rdx
    add     rax, rdx
    jmp     .lower_done
    
.lower_all_64_matched:
    add     rax, 64
    add     r9, 64
    jmp     .lower_loop_avx512
    
.lower_scalar:
    ; --------------------------------------------------------
    ; AVX-512 MASKED TAIL (Replaces Scalar Loop)
    ; --------------------------------------------------------
    test    rdx, rdx
    jz      .lower_done
    
    mov     r14, -1
    bzhi    r14, r14, rdx
    kmovq   k1, r14
    
    vmovdqu8 zmm0 {k1}{z}, [r11 + rsi]
    vmovdqu8 zmm1 {k1}{z}, [r12 + r9]
    
    vpcmpeqb k2 {k1}, zmm0, zmm1
    
    kmovq   r14, k2
    not     r14
    bzhi    r14, r14, rdx
    
    test    r14, r14
    jz      .lower_tail_match
    
    tzcnt   r14, r14
    add     rax, r14
    jmp     .lower_done
    
.lower_tail_match:
    add     rax, rdx
    jmp     .lower_done
    
.lower_done:
    ; --- STOP LEFT (LOWER) DIAGONAL TIMER ---
    STOP_TIMER [rbp-48], OFF_LEFT_DIAG

    cmp     rax, rbx
    jbe     .next_diag
    mov     rbx, rax
    
    mov     rcx, r13
    sub     rcx, r15
    cmp     rbx, rcx
    jae     .accept
    
.next_diag:
    inc     r8
    jmp     .diag_loop
    
.diag_done:
    ; index = index + GlobalCount
    add     r15, rbx
    
    ; if (index < ReadLength)
    cmp     r15, r13
    jae     .accept
    
    ; Edits++; index++;
    inc     qword [rbp-16]
    inc     r15
    
    mov     r14, [rbp-16]
    mov     [current_edits], r14
    cmp     r14, r10
    jg      .reject
    
    inc     qword [rbp-24]
    jmp     .while_loop

.accept:
    mov     [current_position], r15
    mov     rax, [rbp-16]
    mov     [current_edits], rax
    cmp     rax, r10
    jg      .reject
    mov     rax, 1
    jmp     .end

.reject:
    xor     eax, eax

.end:
    ; --- STOP TOTAL TIMER ---
    STOP_TIMER [rbp-56], OFF_TOTAL

    ; Clear ZMM registers
    vzeroupper
    
    add     rsp, 64
    pop     r15
    pop     r14
    pop     r13
    pop     r12
    pop     rbx
    leave
    ret

Overwriting fullalignment.asm


In [6]:
%%writefile no_macros_fullalignment.asm

default rel
bits 64

; ==============================================================================
; DISABLE TIMERS FOR PERFORMANCE
; ==============================================================================
%macro START_TIMER 1
%endmacro

%macro STOP_TIMER 2
%endmacro

section .data
global SneakySnake
global current_position
global current_edits
global mismatch_count
global safety_counter

extern timing_data

align 64
current_position dq 0
current_edits dq 0
mismatch_count dq 0
safety_counter dq 0

section .text
global SneakySnake

SneakySnake:
    push    rbp
    mov     rbp, rsp
    push    rbx
    push    r12
    push    r13
    push    r14
    push    r15
    sub     rsp, 64 

    START_TIMER [rbp-56]

    mov     r13, rdi              ; ReadLength
    mov     r12, rsi              ; RefSeq
    mov     r11, rdx              ; ReadSeq
    mov     r10, rcx              ; EditThreshold
    mov     [rbp-8], r8           ; IterationNo

    xor     rax, rax
    mov     [current_position], rax
    mov     [current_edits], rax
    mov     [mismatch_count], rax
    mov     [safety_counter], rax

    xor     r15, r15              ; index = 0
    mov     qword [rbp-16], 0     ; Edits = 0
    mov     qword [rbp-24], 1     ; roundsNo = 1
    
.while_loop:
    cmp     r15, r13
    jae     .accept
    
    mov     rax, [rbp-24]
    cmp     rax, [rbp-8]
    ja      .reject
    
    mov     r14, [rbp-16]
    cmp     r14, r10
    jg      .reject
    
    START_TIMER [rbp-32]

    xor     rbx, rbx              ; GlobalCount = 0
    
    mov     rax, r15
    mov     rcx, r13
    sub     rcx, rax
    
    prefetcht0 [r11 + rax + 64]
    prefetcht0 [r12 + rax + 64]
    
.main_loop_avx512_unrolled:
    cmp     rcx, 256
    jb      .main_loop_avx512_single
    
    prefetcht0 [r11 + rax + 320]
    prefetcht0 [r12 + rax + 320]
    
    vmovdqu8 zmm0, [r11 + rax]
    vmovdqu8 zmm1, [r12 + rax]
    vmovdqu8 zmm2, [r11 + rax + 64]
    vmovdqu8 zmm3, [r12 + rax + 64]
    
    vpcmpeqb k1, zmm0, zmm1
    vpcmpeqb k2, zmm2, zmm3
    
    kmovq   rdx, k1
    not     rdx
    test    rdx, rdx
    jnz     .found_mismatch_0
    
    kmovq   rdx, k2
    not     rdx
    test    rdx, rdx
    jnz     .found_mismatch_64
    
    vmovdqu8 zmm4, [r11 + rax + 128]
    vmovdqu8 zmm5, [r12 + rax + 128]
    vmovdqu8 zmm6, [r11 + rax + 192]
    vmovdqu8 zmm7, [r12 + rax + 192]
    
    vpcmpeqb k3, zmm4, zmm5
    vpcmpeqb k4, zmm6, zmm7
    
    kmovq   rdx, k3
    not     rdx
    test    rdx, rdx
    jnz     .found_mismatch_128
    
    kmovq   rdx, k4
    not     rdx
    test    rdx, rdx
    jnz     .found_mismatch_192
    
    add     rbx, 256
    add     rax, 256
    sub     rcx, 256
    jmp     .main_loop_avx512_unrolled

.found_mismatch_0:
    tzcnt   rdx, rdx
    add     rbx, rdx
    jmp     .main_done

.found_mismatch_64:
    tzcnt   rdx, rdx
    add     rbx, 64
    add     rbx, rdx
    jmp     .main_done

.found_mismatch_128:
    tzcnt   rdx, rdx
    add     rbx, 128
    add     rbx, rdx
    jmp     .main_done

.found_mismatch_192:
    tzcnt   rdx, rdx
    add     rbx, 192
    add     rbx, rdx
    jmp     .main_done
    
.main_loop_avx512_single:
    cmp     rcx, 64
    jb      .main_loop_scalar
    
    vmovdqu8 zmm0, [r11 + rax]
    vmovdqu8 zmm1, [r12 + rax]
    
    vpcmpeqb k1, zmm0, zmm1
    kmovq   rdx, k1
    not     rdx
    
    test    rdx, rdx
    jz      .all_64_matched
    
    tzcnt   rdx, rdx
    add     rbx, rdx
    jmp     .main_done
    
.all_64_matched:
    add     rbx, 64
    add     rax, 64
    sub     rcx, 64
    jmp     .main_loop_avx512_single
    
.main_loop_scalar:
    ; --------------------------------------------------------
    ; AVX-512 MASKED TAIL (Replaces Scalar Loop)
    ; --------------------------------------------------------
    test    rcx, rcx
    jz      .main_done
    
    mov     rdx, -1
    bzhi    rdx, rdx, rcx       ; Create mask for remaining bytes
    kmovq   k1, rdx
    
    vmovdqu8 zmm0 {k1}{z}, [r11 + rax]
    vmovdqu8 zmm1 {k1}{z}, [r12 + rax]
    
    vpcmpeqb k2 {k1}, zmm0, zmm1
    
    kmovq   r8, k2
    not     r8
    and     r8, rdx             ; Clear garbage bits
    
    test    r8, r8
    jz      .main_tail_match
    
    tzcnt   r8, r8
    add     rbx, r8
    jmp     .main_done
    
.main_tail_match:
    add     rbx, rcx
    jmp     .main_done
    
.main_done:
    STOP_TIMER [rbp-32], OFF_MAIN_DIAG

    mov     rcx, r13
    sub     rcx, r15
    cmp     rbx, rcx
    jae     .accept
    
    mov     r8, 1
    
.diag_loop:
    cmp     r8, r10
    ja      .diag_done
    
    START_TIMER [rbp-40]

    xor     rax, rax
    mov     r9, r15
    
.upper_loop_avx512:
    cmp     r9, r8
    jb      .upper_done
    
    mov     rsi, r9
    sub     rsi, r8
    cmp     rsi, r13
    jae     .upper_done
    
    mov     rdx, r13
    sub     rdx, r9
    mov     rdi, r13
    sub     rdi, rsi
    cmp     rdx, rdi
    cmovg   rdx, rdi
    
    cmp     rdx, 128
    jb      .upper_single_vector
    
    prefetcht0 [r11 + rsi + 128]
    prefetcht0 [r12 + r9 + 128]
    
    vmovdqu8 zmm0, [r11 + rsi]
    vmovdqu8 zmm1, [r12 + r9]
    vmovdqu8 zmm2, [r11 + rsi + 64]
    vmovdqu8 zmm3, [r12 + r9 + 64]
    
    vpcmpeqb k1, zmm0, zmm1
    vpcmpeqb k2, zmm2, zmm3
    
    kmovq   rdx, k1
    not     rdx
    test    rdx, rdx
    jnz     .upper_mismatch_0
    
    kmovq   rdx, k2
    not     rdx
    test    rdx, rdx
    jnz     .upper_mismatch_64
    
    add     rax, 128
    add     r9, 128
    jmp     .upper_loop_avx512

.upper_mismatch_0:
    tzcnt   rdx, rdx
    add     rax, rdx
    jmp     .upper_done

.upper_mismatch_64:
    tzcnt   rdx, rdx
    add     rax, 64
    add     rax, rdx
    jmp     .upper_done
    
.upper_single_vector:
    cmp     rdx, 64
    jb      .upper_scalar
    
    vmovdqu8 zmm0, [r11 + rsi]
    vmovdqu8 zmm1, [r12 + r9]
    
    vpcmpeqb k1, zmm0, zmm1
    kmovq   rdx, k1
    not     rdx
    
    test    rdx, rdx
    jz      .upper_all_64_matched
    
    tzcnt   rdx, rdx
    add     rax, rdx
    jmp     .upper_done
    
.upper_all_64_matched:
    add     rax, 64
    add     r9, 64
    jmp     .upper_loop_avx512
    
.upper_scalar:
    ; --------------------------------------------------------
    ; AVX-512 MASKED TAIL (Replaces Scalar Loop)
    ; --------------------------------------------------------
    test    rdx, rdx            ; rdx already holds the safe length
    jz      .upper_done
    
    mov     r14, -1             ; Use r14 temporary (caller-save was pushed)
    bzhi    r14, r14, rdx
    kmovq   k1, r14
    
    vmovdqu8 zmm0 {k1}{z}, [r11 + rsi]
    vmovdqu8 zmm1 {k1}{z}, [r12 + r9]
    
    vpcmpeqb k2 {k1}, zmm0, zmm1
    
    kmovq   r14, k2
    not     r14
    bzhi    r14, r14, rdx       ; Clean garbage bits again
    
    test    r14, r14
    jz      .upper_tail_match
    
    tzcnt   r14, r14
    add     rax, r14
    jmp     .upper_done
    
.upper_tail_match:
    add     rax, rdx
    jmp     .upper_done
    
.upper_done:
    STOP_TIMER [rbp-40], OFF_RIGHT_DIAG

    cmp     rax, rbx
    jbe     .check_lower
    mov     rbx, rax
    
    mov     rcx, r13
    sub     rcx, r15
    cmp     rbx, rcx
    jae     .accept
    
.check_lower:
    START_TIMER [rbp-48]

    xor     rax, rax
    mov     r9, r15
    
.lower_loop_avx512:
    lea     rsi, [r9 + r8]
    cmp     rsi, r13
    jae     .lower_done
    
    mov     rdx, r13
    sub     rdx, r8
    cmp     r9, rdx
    jae     .lower_done
    
    mov     rdx, r13
    sub     rdx, r9
    mov     rdi, r13
    sub     rdi, rsi
    cmp     rdx, rdi
    cmovg   rdx, rdi
    
    cmp     rdx, 128
    jb      .lower_single_vector
    
    prefetcht0 [r11 + rsi + 128]
    prefetcht0 [r12 + r9 + 128]
    
    vmovdqu8 zmm0, [r11 + rsi]
    vmovdqu8 zmm1, [r12 + r9]
    vmovdqu8 zmm2, [r11 + rsi + 64]
    vmovdqu8 zmm3, [r12 + r9 + 64]
    
    vpcmpeqb k1, zmm0, zmm1
    vpcmpeqb k2, zmm2, zmm3
    
    kmovq   rdx, k1
    not     rdx
    test    rdx, rdx
    jnz     .lower_mismatch_0
    
    kmovq   rdx, k2
    not     rdx
    test    rdx, rdx
    jnz     .lower_mismatch_64
    
    add     rax, 128
    add     r9, 128
    jmp     .lower_loop_avx512

.lower_mismatch_0:
    tzcnt   rdx, rdx
    add     rax, rdx
    jmp     .lower_done

.lower_mismatch_64:
    tzcnt   rdx, rdx
    add     rax, 64
    add     rax, rdx
    jmp     .lower_done
    
.lower_single_vector:
    cmp     rdx, 64
    jb      .lower_scalar
    
    vmovdqu8 zmm0, [r11 + rsi]
    vmovdqu8 zmm1, [r12 + r9]
    
    vpcmpeqb k1, zmm0, zmm1
    kmovq   rdx, k1
    not     rdx
    
    test    rdx, rdx
    jz      .lower_all_64_matched
    
    tzcnt   rdx, rdx
    add     rax, rdx
    jmp     .lower_done
    
.lower_all_64_matched:
    add     rax, 64
    add     r9, 64
    jmp     .lower_loop_avx512
    
.lower_scalar:
    ; --------------------------------------------------------
    ; AVX-512 MASKED TAIL (Replaces Scalar Loop)
    ; --------------------------------------------------------
    test    rdx, rdx
    jz      .lower_done
    
    mov     r14, -1
    bzhi    r14, r14, rdx
    kmovq   k1, r14
    
    vmovdqu8 zmm0 {k1}{z}, [r11 + rsi]
    vmovdqu8 zmm1 {k1}{z}, [r12 + r9]
    
    vpcmpeqb k2 {k1}, zmm0, zmm1
    
    kmovq   r14, k2
    not     r14
    bzhi    r14, r14, rdx
    
    test    r14, r14
    jz      .lower_tail_match
    
    tzcnt   r14, r14
    add     rax, r14
    jmp     .lower_done
    
.lower_tail_match:
    add     rax, rdx
    jmp     .lower_done
    
.lower_done:
    STOP_TIMER [rbp-48], OFF_LEFT_DIAG

    cmp     rax, rbx
    jbe     .next_diag
    mov     rbx, rax
    
    mov     rcx, r13
    sub     rcx, r15
    cmp     rbx, rcx
    jae     .accept
    
.next_diag:
    inc     r8
    jmp     .diag_loop
    
.diag_done:
    add     r15, rbx
    
    cmp     r15, r13
    jae     .accept
    
    inc     qword [rbp-16]
    inc     r15
    
    mov     r14, [rbp-16]
    mov     [current_edits], r14
    cmp     r14, r10
    jg      .reject
    
    inc     qword [rbp-24]
    jmp     .while_loop

.accept:
    mov     [current_position], r15
    mov     rax, [rbp-16]
    mov     [current_edits], rax
    cmp     rax, r10
    jg      .reject
    mov     rax, 1
    jmp     .end

.reject:
    xor     eax, eax

.end:
    STOP_TIMER [rbp-56], OFF_TOTAL
    vzeroupper
    add     rsp, 64
    pop     r15
    pop     r14
    pop     r13
    pop     r12
    pop     rbx
    leave
    ret

Overwriting no_macros_fullalignment.asm


In [4]:
!nasm -f elf64 fullalignment.asm -o fullalignavx.o 
!gcc -c fullalignment.c -o fullalign.o -mavx512f -mavx512bw 
!gcc fullalignavx.o fullalign.o -o simple -mavx512f -mavx512bw 
!./simple "../THES3/DNAPAIRS/ERR240727_1_E2_30000Pairs.txt" 10 100 30000 0

Loading sequences (Fixed Len=100, Limit=30000)...
Loaded 30000 sequences...
=== DEBUGGING FIRST 0 SEQUENCES ===

=== RUNNING FULL BENCHMARK ===

                  BENCHMARK SUMMARY               
Total Pairs              : 30000
Fixed Length             : 100
Threshold                : 10

C Implementation:
  Accepted               : 14623
  Time                   : 0.0609 seconds

ASM Implementation:
  Accepted               : 14623
  Time                   : 0.0973 seconds
  Speedup vs C           : 0.63x

=== ASM INTERNAL TIMING BREAKDOWN ===
Component                    Total (s)        Calls      Avg Cycles
----------------------------------------------------------------------------
Total SneakySnake             0.058490        30000            7799
  Main Diagonal               0.001712       270333              25
  Right (Upper) Diag          0.013439      2588908              21
  Left (Lower) Diag           0.013470      2587750              21
  Other/Overhead              0

In [60]:
!nasm -f elf64 no_macros_fullalignment.asm -o no_macros_fullalignavx.o 
!gcc -c fullalignment.c -o fullalign.o -mavx512f -mavx512bw 
!gcc no_macros_fullalignavx.o fullalign.o -o simple1 -mavx512f -mavx512bw 
!./simple1 "../THES3/DNAPAIRS/ERR240727_1_E2_30000Pairs.txt" 0 100 30000 0

Loading sequences (Fixed Len=100, Limit=30000)...
Loaded 30000 sequences...
=== DEBUGGING FIRST 0 SEQUENCES ===

=== RUNNING FULL BENCHMARK ===

                  BENCHMARK SUMMARY               
Total Pairs              : 30000
Fixed Length             : 100
Threshold                : 0

C Implementation:
  Accepted               : 243
  Time                   : 0.0031 seconds

ASM Implementation:
  Accepted               : 243
  Time                   : 0.0021 seconds
  Speedup vs C           : 1.46x

=== ASM INTERNAL TIMING BREAKDOWN ===
Component                    Total (s)        Calls      Avg Cycles
----------------------------------------------------------------------------
Total SneakySnake             0.000000            0               0
  Main Diagonal               0.000000            0               0
  Right (Upper) Diag          0.000000            0               0
  Left (Lower) Diag           0.000000            0               0
------------------------------------

In [11]:
!nasm -f elf64 fullalignment.asm -o fullalignavx.o 
!gcc -c fullalignment.c -o fullalign.o -mavx512f -mavx512bw 
!gcc fullalignavx.o fullalign.o -o simple -mavx512f -mavx512bw 
!./simple "../THES3/DNAPAIRS/Ecoli_Reads_10bp.txt" 10 10 30000 0

Loading sequences (Fixed Len=10, Limit=30000)...
Loaded 30000 sequences...
=== DEBUGGING FIRST 0 SEQUENCES ===

=== RUNNING FULL BENCHMARK ===

                  BENCHMARK SUMMARY               
Total Pairs              : 30000
Fixed Length             : 10
Threshold                : 10

C Implementation:
  Accepted               : 30000
  Time                   : 0.0186 seconds

ASM Implementation:
  Accepted               : 30000
  Time                   : 0.0447 seconds
  Speedup vs C           : 0.42x

=== ASM INTERNAL TIMING BREAKDOWN ===
Component                    Total (s)        Calls      Avg Cycles
----------------------------------------------------------------------------
Total SneakySnake             0.027483        30000            3664
  Main Diagonal               0.000714       125727              23
  Right (Upper) Diag          0.006097      1125764              22
  Left (Lower) Diag           0.006295      1112023              23
  Other/Overhead              0.0

In [ ]:
!nasm -f elf64 fullalignment.asm -o fullalignavx.o 
!gcc -c fullalignment.c -o fullalign.o -mavx512f -mavx512bw 
!gcc fullalignavx.o fullalign.o -o simple -mavx512f -mavx512bw 
!./simple "../THES3/DNAPAIRS/SRR826471_1_E8_30million.txt" 0 250 30000000 0

Loading sequences (Fixed Len=250, Limit=30000000)...
Loaded 172000 sequences...